In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.influence_func_cp import (
    InfluenceFunctionConformalPredictor,
)

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    lam=0.5,
    kernel="laplacian",
    solver="lbfgs",
    loss_name=loss_name,
    loss_params=loss_params,
)

Instantiate region predictor

In [7]:
conformal_predictor = InfluenceFunctionConformalPredictor(
    predictor, non_conformity_name="absolute"
)
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points, test_input_points
)

In [8]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [9]:
prediction_regions

[{'upper': [-1.6119158249258907,1.6019284815594408],
  'lower': [-1.6118007732180377,1.6018148556126488]},
 {'upper': [-1.6105489341297075,1.6033005646596807],
  'lower': [-1.6104340974046745,1.603186724049397]},
 {'upper': [-1.6115927883473138,1.6022519097846755],
  'lower': [-1.6114777825583317,1.6021382380456277]},
 {'upper': [-1.6114422722521105,1.6023993388966455],
  'lower': [-1.6113272903794786,1.6022856434498727]},
 {'upper': [-1.6126586832665242,1.6012146430877143],
  'lower': [-1.6125435223539502,1.601101124749582]},
 {'upper': [-1.6125109308917456,1.6013662261933503],
  'lower': [-1.6123957883084847,1.6012526894385992]},
 {'upper': [-1.6153795169671477,1.5984710833883313],
  'lower': [-1.6152639313305992,1.5983579889146116]},
 {'upper': [-1.6103826279549514,1.603473879716043],
  'lower': [-1.6102678321932165,1.6033599979439914]},
 {'upper': [-1.6111685496992112,1.602702834900654],
  'lower': [-1.6110536101318214,1.6025890959943974]},
 {'upper': [-1.6103056845266248,1.6035594

In [10]:
coverage_upper = np.mean(
    [
        test_output_point in prediction_region["upper"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_upper)

test coverage:  0.856


In [11]:
coverage_lower = np.mean(
    [
        test_output_point in prediction_region["lower"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_lower)

test coverage:  0.856
